<img src="logo.png" alt="Vegeta" width="240">

# Myropods: Cleopatra — the agile three-segment walker

The same design file as Persephone (`designs/myropod.py`), with three segments instead of twelve and
everything scaled up: a 0.65 m, 12-legged reconnaissance walker — the sheet's *agile walker with limb
sacrifice* — 150 × 110 × 70 mm segments, four legs each, no tether, radio and battery instead. Each
segment stands on its own four legs, so the controller can lose a leg (or a whole segment's worth of
support) and keep walking: that reconfiguration is the point of this notebook.

```
CAD (Dedalus): the Persephone design with Cleopatra's parameters ─► masses against the sheet, envelope
standing, walking 1.2 m/s, crawling 0.4 m/s ─► foot forces (duty factor) ─► joint torques ─► actuator class
limb loss ─► support polygon of a segment on three legs, loads on the survivors, the joint carrying a lame segment
operating cases ─► FEA of the leg (Talos), joint pin by hand ─► leg modes vs stride harmonics
battery module ─► endurance per mode (2–8 h on the sheet)
the dog's gait machine (designs/gait.py) with three bodies ─► twelve legs on open terrain, joint by joint, body joints ─► movie
print a leg (Mellonia) ─► JSON record
```

In [ ]:
import json, math, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from vegeta import dedalus, talos, chronos, mellonia
from vegeta.dedalus import viz as dviz
from vegeta.talos import viz as tviz
from vegeta.mellonia import viz as mviz
from vegeta.mellonia.examples import GENERIC_PLA_0_2MM
import sys; sys.path.insert(0, "designs"); import actuators as act          # the shared actuator / motor / joint library (designs/actuators.py)
import gait                                                                    # the shared gait simulation (designs/gait.py), first written for the dog
import pyvista as pv
from vegeta.talos.viz import _write_video

RUNS = Path("_runs/cleopatra"); shutil.rmtree(RUNS, ignore_errors=True); RUNS.mkdir(parents=True)
design_file = RUNS / "myropod.py"
shutil.copy(Path("designs/myropod.py"), design_file)
myropod = dedalus.load_design(f"{design_file}:Myropod")
CLEO = dict(n_segments=3, seg_length=150.0, seg_width=110.0, seg_height=70.0, shell_thickness=3.0, joint_gap=20.0, joint_diameter=40.0, joint_pin_diameter=10.0,
            femur_length=80.0, tibia_length=100.0, leg_diameter=16.0, foot_diameter=24.0, hip_angle_deg=40.0, knee_angle_deg=85.0, head_length=130.0, tether=False)
p = myropod.resolve(**CLEO)
pd.DataFrame(myropod.params.table()).set_index("name").assign(cleopatra=pd.Series(CLEO)).dropna(subset=["cleopatra"])

## 1. The walker

In [ ]:
cleo = myropod.generate(**CLEO)
seg, leg, head = myropod.generate(part="segment", **CLEO), myropod.generate(part="leg", **CLEO), myropod.generate(part="head", **CLEO)
cad = {k: g.export(RUNS / "cad" / k, stl_tolerance=0.05) for k, g in (("cleopatra", cleo), ("segment", seg), ("leg", leg), ("head", head))}
N = p["n_segments"]; REACH = myropod.leg_reach(p); PITCH = myropod.pitch_length(p)
sheet = pd.DataFrame({"sheet": {"length [m]": "0.55–0.70", "width [m]": "0.22–0.30", "height [m]": "0.10–0.16", "mass [kg]": "3.5–7.5", "legs": "12 (6 x 2)", "walk [m/s]": "to 1.2", "crawl [m/s]": "to 0.4", "run time [h]": "2–8"},
                      "this design": {"length [m]": cleo.dimensions[0] / 1000, "width [m]": cleo.dimensions[1] / 1000, "height [m]": cleo.dimensions[2] / 1000, "mass [kg]": "see below", "legs": f"{4 * N} ({N} x 4)", "walk [m/s]": 1.2, "crawl [m/s]": 0.4, "run time [h]": "see §6"}})
display(sheet)
dviz.show(dviz.plot3d(cleo))

In [ ]:
fig = dviz.plot_sections(cleo, normal="x", positions=[-p["head_length"] / 2 - p["joint_gap"] - p["seg_length"] / 2 - myropod.hip_x(p)], cols=1)   # through the first segment's front leg pair
RHO_PA12CF = 1.1e-3
ACTUATOR = act.get("smart servo 6 Nm").as_dict()        # legs, from the shared library
JOINT_ACT = act.get("smart servo 12 Nm").as_dict()      # pitch + yaw per body joint
per_segment = pd.Series({"shell + hatch + rib (from CAD)": seg.volume * RHO_PA12CF, "legs 4x (from CAD)": 4 * leg.volume * RHO_PA12CF,
                         "leg servos 12x (hip yaw, hip pitch, knee per leg)": 12 * ACTUATOR["mass_g"], "joint servos 2x": 2 * JOINT_ACT["mass_g"], "segment PCB, IMU, contact sensors, bus": 60.0, "wiring, pins, pads": 50.0})
head_g = head.volume * RHO_PA12CF + 180.0 + 120.0 + 90.0 + 60.0                   # 4K day camera, NIR night camera, LWIR, microphones + IMU + radio
battery_g = 4 * 2 * 48.0 + 60.0                                                     # 4S2P 21700 cells + BMS, in the middle segment
compute_g = 150.0
M_CLEO = (N * per_segment.sum() + head_g + battery_g + compute_g) / 1000
G = 9.81; W = M_CLEO * G
print(f"Cleopatra {M_CLEO:.2f} kg (sheet 3.5–7.5) | {per_segment.sum():.0f} g per segment | stance width {REACH['stance_width']:.0f} mm | belly clearance {REACH['drop_z']:.0f} mm")
per_segment.round(0)

## 2. Standing, walking, crawling: foot forces and joint torques

Each segment carries its own share of the weight on its four legs (the head and battery ride on the
first and middle segments). Gaits per segment: a **crawl** (wave gait, duty β = 0.75, one leg of four
in the air), a **walk** (β = 0.5, two legs down per segment, trot-like) and a **run** (β = 0.35). The
half-sine stance gives the peak per foot `F = (π/2) · W_seg / (4β)`. A leg's hip-pitch servo holds the
foot force over its horizontal reach, the knee over the tibia lever, plus the longitudinal share when
braking or climbing.

In [ ]:
GAITS = {"crawl 0.4 m/s": (0.75, 0.4), "walk 1.2 m/s": (0.50, 1.2), "run 2.0 m/s": (0.35, 2.0)}
W_SEG = {"segment 1 (head)": (per_segment.sum() + head_g + compute_g) / 1000 * G, "segment 2 (battery)": (per_segment.sum() + battery_g) / 1000 * G, "segment 3": per_segment.sum() / 1000 * G}
def foot_peak(W_seg, beta, legs=4): return (math.pi / 2) * W_seg / (legs * beta)
def torques(Fz, Fx, hip=p["hip_angle_deg"], knee=p["knee_angle_deg"]):
    a, b = math.radians(hip), math.radians(knee)
    y_knee, z_knee = p["femur_length"] * math.cos(a), -p["femur_length"] * math.sin(a)
    y_foot, z_foot = y_knee + p["tibia_length"] * math.cos(b), z_knee - p["tibia_length"] * math.sin(b)
    tau_hip = (Fz * y_foot) / 1000; tau_knee = (Fz * (y_foot - y_knee)) / 1000          # vertical force over the horizontal levers
    tau_hip_roll = (Fx * abs(z_foot)) / 1000                                               # the longitudinal force on the leg's vertical drop (hip yaw/roll servo)
    return {"tau_hip_Nm": tau_hip, "tau_knee_Nm": tau_knee, "tau_long_Nm": tau_hip_roll}
rows = {}
for gname, (beta, v) in GAITS.items():
    for sname, Ws in W_SEG.items():
        F = foot_peak(Ws, beta); Fx = 0.3 * F
        rows[f"{gname}, {sname}"] = {"W_seg [N]": Ws, "duty": beta, "peak foot [N]": F, **torques(F, Fx)}
tq = pd.DataFrame(rows).T
tq["servo SF"] = ACTUATOR["stall_Nm"] / tq[["tau_hip_Nm", "tau_knee_Nm", "tau_long_Nm"]].max(axis=1)
tq.round(2)

## 3. Limb loss: reconfiguration

The sheet's *ofiara kończyn*: a leg is lost (jammed, torn off, a servo dead) and the walker reconfigures.
Three questions per segment: can it still **stand** — the segment's centre of gravity must stay inside
the polygon of the remaining feet, with a margin (on three legs of a rectangle the CG sits exactly on the
diagonal, so the controller first shifts the body towards the survivors by re-planting the stance); what the **survivors carry** — the static and the gait
peak per remaining foot; and when a segment has too few legs, can the **joint carry it** — the neighbour
holds the lame segment as a cantilever over the joint pin, a moment the joint's pitch servo must hold.

In [ ]:
hx, sw = myropod.hip_x(p), REACH["stance_width"] / 2 - p["foot_diameter"] / 2
FEET = {"front-left": (hx, sw), "front-right": (hx, -sw), "rear-left": (-hx, sw), "rear-right": (-hx, -sw)}
BODY_SHIFT = 40.0                                     # how far the legs can shift the segment's body over its feet [mm] (input: re-planting the stance)
def polygon_margin(feet, cg=(0.0, 0.0)):
    """Distance from the segment's CG to the nearest edge of the convex hull of the feet; negative = outside."""
    pts = np.array(feet); cg = np.array(cg)
    if len(pts) < 3:
        return float("-inf")
    c = pts.mean(axis=0); order = np.argsort(np.arctan2(pts[:, 1] - c[1], pts[:, 0] - c[0])); pts = pts[order]
    best = float("inf")
    for i in range(len(pts)):
        a, b = pts[i], pts[(i + 1) % len(pts)]
        edge = b - a; n = np.array([edge[1], -edge[0]]) / np.linalg.norm(edge)
        inside = np.dot(c - a, n)
        best = min(best, np.dot(cg - a, n) * np.sign(inside))
    return float(best)
def best_margin(feet):
    """The margin after the controller shifts the body (up to BODY_SHIFT) towards the centroid of the remaining feet."""
    if len(feet) < 3:
        return float("-inf")
    c = np.mean(feet, axis=0); d = np.linalg.norm(c)
    cg = c * min(1.0, BODY_SHIFT / d) if d > 1e-9 else np.zeros(2)
    return polygon_margin(feet, cg)
loss = {}
Ws = W_SEG["segment 2 (battery)"]
for lost in ([], ["front-left"], ["front-left", "rear-right"], ["front-left", "rear-left"], ["front-left", "front-right", "rear-left"]):
    remaining = [v for k, v in FEET.items() if k not in lost]
    n_left = len(remaining)
    margin = best_margin(remaining)
    stand = Ws / n_left if n_left else float("nan")
    crawl_peak = foot_peak(Ws, 0.75 * 4 / n_left if n_left else 1, legs=n_left) if n_left >= 3 else float("nan")   # a crawl keeps one leg up at a time
    moment = 0.0 if margin > 0 else Ws * PITCH / 2 / 1000                                                            # the neighbour carries the segment over the joint
    loss[", ".join(lost) if lost else "all four legs"] = {"legs left": n_left, "stability margin [mm]": margin, "stands alone": margin > 15, "static per foot [N]": stand,
                                                          "crawl peak per foot [N]": crawl_peak, "servo SF (crawl)": ACTUATOR["stall_Nm"] / max(torques(crawl_peak, 0.3 * crawl_peak).values()) if n_left >= 3 else float("nan"),
                                                          "joint moment to carry it [Nm]": moment, "joint SF": JOINT_ACT["stall_Nm"] / moment if moment > 0 else float("inf")}
loss_tab = pd.DataFrame(loss).T
fig, ax = plt.subplots(figsize=(5.5, 5))
for k, (x, y) in FEET.items():
    ax.plot(x, y, "o", ms=10, color="#1565c0" if k != "front-left" else "#c62828"); ax.text(x, y + 6, k, ha="center", fontsize=8)
rem = np.array([v for k, v in FEET.items() if k != "front-left"]); hull = rem[np.argsort(np.arctan2(rem[:, 1], rem[:, 0]))]
ax.fill(hull[:, 0], hull[:, 1], alpha=0.15, color="#2e7d32", label="support on three legs"); ax.plot(0, 0, "k+", ms=12, label="segment CG")
ax.set(aspect="equal", xlabel="x [mm]", ylabel="y [mm]", title=f"one leg lost: margin {polygon_margin(rem):.0f} mm as is, {best_margin(rem):.0f} mm after a {BODY_SHIFT:.0f} mm body shift"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
loss_tab.round(2)

## 4. Strength of the leg (Talos), the joint pin by hand

The leg in its own frame (hip bore held, forces on the foot pad, rotated from the ground frame with the
standing angles): standing, the walk and run peaks, the three-leg crawl after a loss, a 0.2 m drop onto
one segment's legs, and a stuck leg at servo stall.

In [ ]:
PA12CF = talos.Material("PA12-CF (printed)", youngs_modulus=3500.0, poissons_ratio=0.40, density=1.1e-9, yield_strength=60.0, source="nominal datasheet, flat orientation; assume 30 % less across layers")
STEEL_PIN = talos.Material("steel pin", youngs_modulus=210000.0, poissons_ratio=0.30, density=7.85e-9, yield_strength=400.0, source="handbook")
fy, dz, fd = REACH["foot_y"], REACH["drop_z"], p["foot_diameter"]
LEG_REGIONS = [talos.SurfacesInBox("hip", (-12, -6.0, -6.0, 12, 6.0, 6.0)), talos.SurfacesInBox("foot", (-fd, fy - fd, -dz - fd, fd, fy + fd, -dz + fd))]
def leg_model(loads, name, masses=()):
    return talos.StructuralModel(cad["leg"].artifacts["step"], "mm-N-MPa", PA12CF, LEG_REGIONS, [talos.FixedSupport("hip")], loads,
                                 talos.MeshSettings(element_size=3.0, min_element_size=1.0, curvature_points=10), name=name, masses=list(masses))
DROP_M, S_LEG = 0.20, 0.06
F_DROP = Ws * (DROP_M / S_LEG + 1) / 4 * math.pi / 2
F_STALL = ACTUATOR["stall_Nm"] * 1000 / REACH["foot_y"]
run_peak = foot_peak(Ws, GAITS["run 2.0 m/s"][0]); walk_peak = foot_peak(Ws, GAITS["walk 1.2 m/s"][0])
LEG_CASES = {   # forces on the foot pad: fz up (the ground pushes the foot up), fx along the body, fy outward (a side slip)
    "standing": dict(fz=Ws / 4),
    "walk peak": dict(fz=walk_peak, fx=0.3 * walk_peak),
    "run peak": dict(fz=run_peak, fx=0.4 * run_peak),
    "crawl on three legs": dict(fz=loss_tab.loc["front-left", "crawl peak per foot [N]"], fx=0.3 * loss_tab.loc["front-left", "crawl peak per foot [N]"]),
    "drop 0.2 m": dict(fz=F_DROP, fx=0.2 * F_DROP),
    "stuck leg at servo stall": dict(fy=-F_STALL, fz=0.5 * F_STALL),
}
leg_model([talos.Force("foot", fz=1.0)], "mesh").mesh(RUNS / "leg_mesh", progress=True)
leg_results = {}
for name, f in tqdm(LEG_CASES.items()):
    key = "leg_" + name.replace(" ", "_").replace(".", "")
    mdl = leg_model([talos.Force("foot", **f)], key); mdl.mesh(RUNS / key); leg_results[name] = mdl.solve(RUNS / key)
def pin_stress(F, d): return math.sqrt(3) * F / (2 * math.pi * d**2 / 4)
F_JOINT = loss_tab.loc["front-left, rear-left", "joint moment to carry it [Nm]"] * 1000 / (p["joint_gap"] / 2) + Ws       # the pin force under the cantilever moment (couple over the neck) plus the weight
static = pd.concat([pd.DataFrame({f"leg — {k}": {"load": ", ".join(f"{kk} {vv:.0f} N" for kk, vv in LEG_CASES[k].items()), "max_von_mises_MPa": r.metrics.get("max_von_mises"), "SF_yield": r.metrics.get("safety_factor_yield"), "deflection_mm": r.metrics.get("max_displacement")} for k, r in leg_results.items()}).T,
                    pd.DataFrame({"joint pin — carrying a lame segment": {"load": f"{F_JOINT:.0f} N shear", "max_von_mises_MPa": pin_stress(F_JOINT, p["joint_pin_diameter"]), "SF_yield": STEEL_PIN.yield_strength / pin_stress(F_JOINT, p["joint_pin_diameter"]), "deflection_mm": float("nan")}}).T])
fig, ax = plt.subplots(figsize=(8, 4))
static["SF_yield"].astype(float).plot.barh(ax=ax, color="#1565c0"); ax.axvline(1.5, color="#c62828", ls="--", label="SF 1.5"); ax.set(xscale="log", xlabel="safety factor on yield"); ax.legend(); ax.grid(alpha=0.3, axis="x")
display(static.round(2))
worst = static.loc[[k for k in static.index if k.startswith("leg")], "SF_yield"].astype(float).idxmin().split(" — ")[1]
tviz.show(tviz.plot_results(leg_results[worst], field="von_mises"))

## 5. Leg modes against the stride harmonics

In [ ]:
modal = leg_model([], "leg_modal", masses=[talos.PointMass("foot", 0.012e-3)])
modal.mesh(RUNS / "leg_modal"); modes = modal.solve_modes(RUNS / "leg_modal", n_modes=3)
freqs = modes.metrics["frequencies_hz"]
speeds = np.linspace(0.2, 2.2, 30)
STRIDE_M = 0.28                                                                      # stride length at a walk [m] (input: ~ two leg reaches)
f_stride = speeds / STRIDE_M
fig, ax = plt.subplots(figsize=(8, 4.5))
for k in (1, 2, 4, 8):
    ax.plot(speeds, k * f_stride, label=f"stride x{k}")
for i, f in enumerate(freqs):
    ax.axhline(f, color="#c62828", ls="--", lw=0.8); ax.text(speeds[0], f, f" leg mode {i + 1}: {f:.0f} Hz", color="#c62828", va="bottom", fontsize=8)
for name, (beta, v) in GAITS.items():
    ax.axvline(v, color="#888", ls=":"); ax.text(v, 0.4, f" {name}", rotation=90, va="bottom", fontsize=8)
ax.set(xlabel="speed [m/s]", ylabel="frequency [Hz]", yscale="log", ylim=(0.3, 2000), title="frequency diagram: stride harmonics vs the leg modes"); ax.legend(fontsize=7, loc="upper left"); ax.grid(alpha=0.3, which="both")
print(f"leg modes {[round(f, 1) for f in freqs]} Hz | stride at a walk {1.2 / STRIDE_M:.1f} Hz")
tviz.show(tviz.plot_mode(modes, mode=1))

## 6. Endurance

A 4S2P 21700 pack (about 140 Wh) in the middle segment. Walking power from a cost of transport
(`COT × W × v`, COT ≈ 1.5 for small legged robots) plus the servos' holding losses and the electronics;
a stationary sentinel only listens.

In [ ]:
E_WH = 8 * 3.6 * 4.8 * 0.9                                                           # 8 cells x 3.6 V x 4.8 Ah, 90 % usable
COT = 1.5; P_ELEC = 9.0; P_HOLD = 4 * N * 0.6                                       # electronics; servos holding the stance [W]
modes_p = {"sentinel (standing, sensors on)": P_ELEC + 0.3 * P_HOLD, "crawl 0.4 m/s": P_ELEC + P_HOLD + COT * W * 0.4 / 0.6, "walk 1.2 m/s": P_ELEC + P_HOLD + COT * W * 1.2 / 0.6, "run 2.0 m/s": P_ELEC + P_HOLD + COT * W * 2.0 / 0.5}
endurance = pd.DataFrame({k: {"power [W]": v, "hours": E_WH / v, "range [km]": E_WH / v * float(k.split(" ")[1]) * 3.6 if "m/s" in k else float("nan")} for k, v in modes_p.items()}).T
print(f"pack {E_WH:.0f} Wh usable")
endurance.round(2)

## 7. Cleopatra's gait on open terrain: twelve legs, joint by joint, as a movie

The robot dog's stepping machine (`designs/gait.py`, written for notebook 16) with a Myropod's bodies and
legs. Every **segment is its own body** on its own four legs: it rides at its standing height over the
ground under its hips and pitches and rolls with it, exactly like the dog's single body. Each segment
walks a lateral-sequence wave (rear-left, front-left, rear-right, front-right, duty 0.75: three feet down
per segment at all times), and the segments are shifted by a twelfth of a stride from the tail to the
head, so a **metachronal wave** runs forward along the body like a millipede's. A Myropod leg has
**hip yaw** (the sweep forward and back), **hip pitch** (the femur below horizontal) and the **knee** (the
tibia below horizontal); its feet stand out to the side at the standing reach, not under the hip.

On top of the legs, the **body joints** between segments: neighbouring segments pitch and roll
independently over the ground, and the two-axis joint (pitch + yaw) must take the difference — its pitch
servo the relative pitch, and the relative roll, which it has no axis for, goes through the neck as a
twist. Step table, joint traces, body-joint angles, the forward-kinematics check, then the movie
(`output/18_myropod_cleopatra/cleopatra_walk_terrain.mp4`).

In [ ]:
terrain = gait.make_terrain(seed=5, rock_height=(15.0, 40.0), rock_width=(50.0, 90.0), waves=((25.0, 1800.0, 0.0), (10.0, 600.0, 0.6)))   # rocks to Cleopatra's scale
V_C, BETA_C = 0.2, 0.75                       # walking speed on rough ground [m/s] and duty factor (inputs)
STRIDE_C = 110.0                              # stride [mm] (input: keeps the hip yaw inside ±45°)
T_C = STRIDE_C / (V_C * 1000)
H_HIP = REACH["drop_z"]; FOOT_OUT = REACH["foot_y"]; HIP_Y = p["seg_width"] / 2 + 6; HX = myropod.hip_x(p)
X0, DT_C, T_END_C = 600.0, 1 / 20, 4.0
times_c = np.arange(0, T_END_C, DT_C)
SEGS = [f"segment {i + 1}" for i in range(N)]
def chain_pose(t_s):
    """Every segment over the ground on its own: segment 1 behind the head, the rest at one pitch each."""
    return [gait.body_pose(X0 + V_C * 1000 * t_s - i * PITCH, HX, HIP_Y, H_HIP, terrain) for i in range(N)]
BASE = {"rear-left": 0.0, "front-left": 0.25, "rear-right": 0.5, "front-right": 0.75}
LEGS_C = {f"s{i + 1} {k}": (i, sx, sy) for i in range(N) for k, (sx, sy) in gait.LEGS.items()}
PHASES_C = {f"s{i + 1} {k}": (BASE[k] + (N - 1 - i) / 12) % 1.0 for i in range(N) for k in gait.LEGS}      # the wave runs from the tail to the head
JN = ("hip yaw", "hip pitch", "knee")
walk_c = gait.simulate_steps(times_c, chain_pose, LEGS_C, PHASES_C, BETA_C, STRIDE_C, T_C, terrain,
                             lambda rel, sx, sy: gait.ik_myropod(rel, p["femur_length"], p["tibia_length"], sy), swing_lift=40.0,
                             torque_fn=lambda ang, rel, F, sx, sy: gait.torques_myropod(ang, rel, F, p["femur_length"], p["tibia_length"], sy),
                             force_body_fn=lambda R, foot: R.T @ np.array([0.0, 0.0, max(W_SEG.values()) / 3]),     # the heaviest segment's weight on three feet
                             joint_names=JN, peak_Nm=ACTUATOR["stall_Nm"], body_names=SEGS, foot_out=FOOT_OUT)
# forward-kinematics check: the joint angles must put every reachable foot back where the stepping machine put it
fk_err = 0.0
for leg, (b_, sx, sy) in LEGS_C.items():
    for k, t_s in enumerate(times_c):
        c_, R_, *_r, hips_ = chain_pose(t_s)[b_]
        foot_fk = hips_[gait._key(sx, sy)] + R_ @ gait.fk_myropod(walk_c.joints[leg][k], p["femur_length"], p["tibia_length"], sy)[1]
        rel_ = R_.T @ (walk_c.feet[leg][k] - hips_[gait._key(sx, sy)])
        if gait.ik_myropod(rel_, p["femur_length"], p["tibia_length"], sy)[1]:
            fk_err = max(fk_err, float(np.linalg.norm(foot_fk - walk_c.feet[leg][k])))
# body joints: relative pitch and roll of neighbouring segments, and the stretch of the neck
bj = []
for t_s in times_c:
    poses = chain_pose(t_s); row = {"t": t_s}
    for i in range(N - 1):
        (c0, R0, th0, ph0, _), (c1, R1, th1, ph1, _) = poses[i], poses[i + 1]
        chord = math.degrees(math.atan2(c0[2] - c1[2], c0[0] - c1[0]))
        row[f"joint {i + 1}-{i + 2} pitch"] = math.degrees(th0 - th1); row[f"joint {i + 1}-{i + 2} twist (roll)"] = math.degrees(ph0 - ph1)
        row[f"joint {i + 1}-{i + 2} neck slope"] = chord - math.degrees((th0 + th1) / 2)
    bj.append(row)
bj = pd.DataFrame(bj).set_index("t")
steps_c = walk_c.steps
print(f"Cleopatra on open terrain: {V_C} m/s, stride {STRIDE_C:.0f} mm at {1 / T_C:.2f} Hz, {len(steps_c)} steps of {len(LEGS_C)} legs in {T_END_C:.0f} s, "
      f"{walk_c.unreachable} unreachable samples of {len(times_c) * len(LEGS_C)} | FK check: max {fk_err:.3f} mm")
print(f"body joints: pitch up to {bj.filter(like='pitch').abs().max().max():.1f}° (range ±45°), twist up to {bj.filter(like='twist').abs().max().max():.1f}° (no roll axis: through the neck)")
for j, n in enumerate(JN):
    print(f"{n}: {min(walk_c.joints[l][:, j].min() for l in LEGS_C):.0f}° … {max(walk_c.joints[l][:, j].max() for l in LEGS_C):.0f}°")
steps_c.round(2)

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 12), gridspec_kw={"height_ratios": [2, 2, 3]})
xs_ = np.linspace(X0 - N * PITCH - 200, X0 + V_C * 1000 * T_END_C + 300, 400)
ax = axes[0, 0]
ax.fill_between(xs_, -80, terrain(xs_, HIP_Y + FOOT_OUT), color="#8d7b68", alpha=0.6, label="ground under the left feet")
for i, col in zip(range(N), ("#1565c0", "#2e7d32", "#c62828")):
    f_ = walk_c.feet[f"s{i + 1} front-left"]; ax.plot(f_[:, 0], f_[:, 2], color=col, lw=1.1, label=f"segment {i + 1} front-left foot")
    cen = np.array([chain_pose(t_)[i][0] for t_ in times_c]); ax.plot(cen[:, 0], cen[:, 2], "--", color=col, lw=0.8)
ax.set(ylabel="z [mm]", title="foot paths over the ground (dashed: segment centres)"); ax.legend(fontsize=7); ax.set_aspect("equal"); ax.grid(alpha=0.3)
ax = axes[0, 1]
for col_ in bj.columns:
    if "neck" not in col_:
        ax.plot(bj.index, bj[col_], "-" if "pitch" in col_ else "--", label=col_)
ax.set(ylabel="deg", title="body joints between segments"); ax.legend(fontsize=7); ax.grid(alpha=0.3)
for ax, seg_i in zip(axes[1], (0, 2)):
    for leg_k, col in (("front-left", "#1565c0"), ("rear-left", "#2e7d32")):
        for j, ls in enumerate(("-.", "-", "--")):
            ax.plot(times_c, walk_c.joints[f"s{seg_i + 1} {leg_k}"][:, j], ls, color=col, label=f"{leg_k}: {JN[j]}")
    ax.set(ylabel="joint angle [deg]", title=f"segment {seg_i + 1}, left legs: all three joints"); ax.legend(fontsize=7, ncol=2); ax.grid(alpha=0.3)
gs = axes[2, 0].get_gridspec(); axes[2, 0].remove(); axes[2, 1].remove()
ax = fig.add_subplot(gs[2, :])
gait.gait_diagram(ax, walk_c, [f"s{i + 1} {k}" for i in range(N) for k in ("front-left", "rear-left", "front-right", "rear-right")])
fig.tight_layout()
fig, ax = plt.subplots(figsize=(12, 3.5))
for j, (n, col) in enumerate(zip(JN, ("#6d4c41", "#1565c0", "#c62828"))):
    ax.plot(steps_c["t [s]"], steps_c[f"τ {n} [Nm]"].abs(), "o", ms=4, color=col, label=n)
ax.axhline(ACTUATOR["stall_Nm"], color="k", ls="-.", lw=0.8, label=f"{ACTUATOR['name']} stall"); ax.axhline(ACTUATOR["rated_Nm"], color="k", ls=":", lw=0.8, label="rated")
ax.set(xlabel="time [s]", ylabel="N m", title="joint torques at each landing (three feet per segment)"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

In [ ]:
OUT = Path("output/18_myropod_cleopatra"); OUT.mkdir(parents=True, exist_ok=True)
seg_mesh = dviz.to_pyvista(seg); head_mesh = dviz.to_pyvista(head)
zc_ = p["seg_height"] / 2
HEAD_DX = (p["head_length"] + p["seg_length"]) / 2 + p["joint_gap"]
gx, gy = np.meshgrid(np.linspace(X0 - N * PITCH - 500, X0 + V_C * 1000 * T_END_C + 600, 140), np.linspace(-450, 450, 70))
ground_c = pv.StructuredGrid(gx, gy, terrain(gx, gy)); ground_c["z"] = terrain(gx, gy).ravel(order="F")
def posed(mesh, R, centre):
    m_ = mesh.copy(); m_.points = (m_.points - np.array([0, 0, zc_])) @ R.T + centre      # the part frame has its hips at z = seg_height / 2
    return m_
def cleo_frame(idx, size=(1100, 650)):
    t_s = times_c[idx]; poses = chain_pose(t_s)
    pl = pv.Plotter(off_screen=True, window_size=size)
    pl.add_mesh(ground_c, scalars="z", cmap="copper", show_scalar_bar=False, smooth_shading=True)
    c1, R1 = poses[0][0], poses[0][1]
    pl.add_mesh(posed(head_mesh, R1, c1 + R1 @ np.array([HEAD_DX, 0, 0])), color="#5a5f66", smooth_shading=True)
    for i, (c_, R_, th_, ph_, hips_) in enumerate(poses):
        pl.add_mesh(posed(seg_mesh, R_, c_), color="#5a5f66", smooth_shading=True)
        for k, (sx, sy) in gait.LEGS.items():
            leg = f"s{i + 1} {k}"; hip = hips_[k]
            knee_b, foot_b = gait.fk_myropod(walk_c.joints[leg][idx], p["femur_length"], p["tibia_length"], sy)
            knee_w, foot_w = hip + R_ @ knee_b, hip + R_ @ foot_b
            col = "#2b2e31" if walk_c.stance[leg][idx] else "#a9ae98"
            for a_, b_, r_ in ((hip, knee_w, p["leg_diameter"] / 2), (knee_w, foot_w, p["leg_diameter"] * 0.4)):
                pl.add_mesh(pv.Cylinder(center=(a_ + b_) / 2, direction=b_ - a_, radius=r_, height=float(np.linalg.norm(b_ - a_))), color=col, smooth_shading=True)
            pl.add_mesh(pv.Sphere(radius=p["leg_diameter"] / 2 + 1, center=knee_w), color=col)
            pl.add_mesh(pv.Sphere(radius=p["foot_diameter"] / 2, center=foot_w), color=col)
    pl.add_text(f"Cleopatra, {V_C} m/s on open terrain   t = {t_s:4.2f} s", font_size=10)
    cm = poses[1][0]
    pl.camera_position = [(cm[0] + 700, cm[1] - 1100, cm[2] + 520), (cm[0] + 40, 0, cm[2] - 80), (0, 0, 1)]
    img = pl.screenshot(return_img=True); pl.close()
    return img
frames_c = [cleo_frame(i) for i in tqdm(range(len(times_c)))]
movie_c = _write_video(frames_c, OUT / "cleopatra_walk_terrain.mp4", int(round(1 / DT_C)))
fig, axes = plt.subplots(2, 3, figsize=(15, 6))
for ax, j in zip(axes.flat, np.linspace(0, len(frames_c) - 1, 6).astype(int)):
    ax.imshow(frames_c[j]); ax.set_axis_off(); ax.set_title(f"t = {times_c[j]:.2f} s")
fig.suptitle("Cleopatra on open terrain (dark legs on the ground, light legs swinging)"); fig.tight_layout()
print(f"movie: {movie_c} ({len(frames_c)} frames, {T_END_C:.0f} s)")

## 8. Print a leg (Mellonia) and export

In [ ]:
prn = mellonia.slice_stl(cad["leg"].artifacts["stl"], GENERIC_PLA_0_2MM, mellonia.Orientation(rotate_y=90), RUNS / "print_leg")   # the leg lies flat on its side
print(prn)
if prn.ok:
    mviz.show(mviz.plot_toolpath(prn))

In [ ]:
summary_doc = {"family": "Myropods", "member": "Cleopatra", "design": {"file": "designs/myropod.py", "parameters": p}, "mass_kg": M_CLEO, "per_segment_g": per_segment.to_dict(),
               "actuators": {"leg": ACTUATOR, "joint": JOINT_ACT}, "envelope_mm": cleo.dimensions, "gaits": tq.to_dict(orient="index"), "limb_loss": loss_tab.to_dict(orient="index"),
               "static": static.to_dict(orient="index"), "leg_modes_hz": freqs, "endurance": endurance.to_dict(orient="index"),
               "terrain_walk": {"speed_m_s": V_C, "stride_mm": STRIDE_C, "steps": steps_c.to_dict(orient="index"), "unreachable_samples": walk_c.unreachable,
                                "fk_check_mm": fk_err, "body_joints_max_deg": bj.abs().max().to_dict(), "movie": str(movie_c)}}
(RUNS / "cleopatra.json").write_text(json.dumps(summary_doc, indent=2, default=str))
print("written:", sorted(x.name for x in RUNS.iterdir()))

**Next steps an engineer would take:** the height is above the sheet's 0.10–0.16 m — a lower stance
(hip 55°, knee 95°) brings the belly down at the price of leg reach, worth a parameter study; measure
the smart servos' real stall and holding currents (the endurance hangs on them); and try the three-leg
crawl on a real segment before trusting the 15 mm stability margin.